## Training script



In [5]:
# Jupyter notebook auto reload
%load_ext autoreload
%autoreload 2

import json
import torch
import torch.nn as nn
import torch.optim as optim
from char_tokenizer import TinyArithmeticCharTokenizer
from tiny_transformer import TinyArithmeticTransformer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
tokenizer = TinyArithmeticCharTokenizer()

with open("dataset.json", "r") as f:
    raw_data = json.load(f)

tokenized_sequences = []
for expression, result in raw_data:
    full_str = f"{expression}={result}<eos>"
    encoded_ids = tokenizer.encode(full_str)
    tokenized_sequences.append(encoded_ids)

max_seq_len = 256
pad_id = tokenizer.char_to_int["<pad>"]

padded_sequences = []
for seq in tokenized_sequences:
    if len(seq) < max_seq_len:
        seq = seq + [pad_id] * (max_seq_len - len(seq))
    else:
        seq = seq[:max_seq_len]
    padded_sequences.append(seq)

dataset_tensor = torch.tensor(padded_sequences, dtype=torch.long)

In [8]:
print(f"Dataset securely loaded in-memory. Matrix shape: {dataset_tensor.shape}")
print(dataset_tensor[:1])

Dataset securely loaded in-memory. Matrix shape: torch.Size([10000, 256])
tensor([[ 4,  6, 15,  0,  9, 16, 10, 16, 11,  1,  3, 15,  9,  3, 16, 13, 16, 11,
          3,  1, 15,  1, 16, 11, 16,  4,  2, 15,  9, 16, 11, 16, 11,  3,  6, 15,
          3,  4, 16, 10, 16,  4,  9, 15,  9,  9, 16, 12, 16, 11,  9,  4, 15,  1,
          4, 16, 13, 16,  5,  9, 15,  4,  7, 16, 10, 16, 11,  1, 15,  5,  9, 16,
         12, 16, 11,  4,  2, 15,  4,  9, 14,  2,  8, 15,  4, 19, 19, 19, 19, 19,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17,
         17, 17, 17, 17, 17, 17, 17

In [9]:
from torch.utils.data import DataLoader, random_split

total_rows = dataset_tensor.size(0)

train_percent = 0.80
val_percent   = 0.10
test_percent  = 0.10

train_size = int(train_percent * total_rows)
val_size   = int(val_percent * total_rows)
test_size  = total_rows - train_size - val_size

train_subset, val_subset, test_subset = random_split(
    dataset_tensor, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 64

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

print(f"Train Loader: {len(train_loader)} batches (Total: {len(train_subset)} shuffled rows)")
print(f"Val Loader:   {len(val_loader)} batches (Total: {len(val_subset)} shuffled rows)")
print(f"Test Loader:  {len(test_loader)} batches (Total: {len(test_subset)} shuffled rows)")

Train Loader: 125 batches (Total: 8000 shuffled rows)
Val Loader:   16 batches (Total: 1000 shuffled rows)
Test Loader:  16 batches (Total: 1000 shuffled rows)
